In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    mean_squared_log_error
)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr, spearmanr

In [2]:
xgb_base_params = {
    'learning_rate': 0.2,
    'n_estimators': 500,
    'max_depth': 6,
    'subsample': 1,
    'colsample_bytree': 1,
    'reg_alpha': 0,
    'reg_lambda': 1,
    'gamma': 0,
    'booster': 'gbtree',
    'min_child_weight': 1
}

lgb_base_params = {
    'learning_rate': 0.1,
    'n_estimators': 500,
    'num_leaves': 31,
    'max_depth': -1,
    'colsample_bytree': 1,
    'subsample': 1,
    'reg_alpha': 0,
    'reg_lambda': 0,
    'boosting_type': 'gbdt'
}

cat_base_params = {
    'learning_rate': 0.03,
    'iterations': 500,
    'depth': 6,
    'l2_leaf_reg': 3,
    'verbose': False,
    'random_seed': 0
}

xgb_meta_params = {
    'learning_rate': 0.2,
    'n_estimators': 100,
    'max_depth': 6,
    'subsample': 1,
    'colsample_bytree': 1
}

In [3]:
# Load dataset
data = pd.read_csv(r"h16_features_D1a_with_deltas.csv")
X = data.iloc[:, :-1].values  # First 9 columns
y = data.iloc[:, -1].values   # PESQ score

In [4]:
# Initialize
scaler = StandardScaler()
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Store metrics
metrics = {
    "rmse": [], "mse": [], "r2": [], "mae": [], "mape": [], "mase": [],
    "msle": [], "mbe": [], "pcc": [], "srcc": [], "ccc": []
}

X_meta_all = []
y_meta_all = []

In [5]:
for train_index, test_index in kf.split(X):
    # Split
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Base models
    xgb_model = XGBRegressor(**xgb_base_params)
    lgb_model = LGBMRegressor(**lgb_base_params)
    cat_model = CatBoostRegressor(**cat_base_params)

    xgb_model.fit(X_train_scaled, y_train)
    lgb_model.fit(X_train_scaled, y_train)
    cat_model.fit(X_train_scaled, y_train)

    # Base predictions
    pred_xgb = xgb_model.predict(X_test_scaled)
    pred_lgb = lgb_model.predict(X_test_scaled)
    pred_cat = cat_model.predict(X_test_scaled)

    # Store OOF meta features
    fold_meta = np.vstack([pred_xgb, pred_lgb, pred_cat]).T
    X_meta_all.append(fold_meta)
    y_meta_all.append(y_test)

    # Average base predictions
    y_pred = (pred_xgb + pred_lgb + pred_cat) / 3

    # Naive baseline for MASE
    y_naive = np.roll(y_test, 1)
    y_naive[0] = y_test[0]

    # Metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    mase = mae / np.mean(np.abs(y_test - y_naive)) if np.mean(np.abs(y_test - y_naive)) != 0 else np.nan
    try:
        msle = mean_squared_log_error(np.maximum(y_test, 1e-6), np.maximum(y_pred, 1e-6))
    except:
        msle = np.nan
    mbe = np.mean(y_pred - y_test)

    pcc, _ = pearsonr(y_test, y_pred)
    srcc, _ = spearmanr(y_test, y_pred)

    y_mean = np.mean(y_test)
    y_pred_mean = np.mean(y_pred)
    cov = np.mean((y_test - y_mean) * (y_pred - y_pred_mean))
    ccc = (2 * cov) / (np.var(y_test) + np.var(y_pred) + (y_mean - y_pred_mean) ** 2)

    # Append fold metrics
    metrics["rmse"].append(rmse)
    metrics["mse"].append(mse)
    metrics["r2"].append(r2)
    metrics["mae"].append(mae)
    metrics["mape"].append(mape)
    metrics["mase"].append(mase)
    metrics["msle"].append(msle)
    metrics["mbe"].append(mbe)
    metrics["pcc"].append(pcc)
    metrics["srcc"].append(srcc)
    metrics["ccc"].append(ccc)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000585 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8159
[LightGBM] [Info] Number of data points in the train set: 768, number of used features: 32
[LightGBM] [Info] Start training from score 0.817351
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [6]:
# Report average fold results
print("\nBase-Model Ensemble (5-Fold Average Metrics):")
for key in metrics:
    print(f"Average {key.upper()}: {np.mean(metrics[key]):.4f}")


Base-Model Ensemble (5-Fold Average Metrics):
Average RMSE: 0.0847
Average MSE: 0.0072
Average R2: 0.4128
Average MAE: 0.0660
Average MAPE: 8.6325
Average MASE: 1.1073
Average MSLE: 0.0023
Average MBE: 0.0018
Average PCC: 0.6495
Average SRCC: 0.6643
Average CCC: 0.5907


In [7]:
# Prepare for meta-model training
X_meta_all = np.vstack(X_meta_all)
y_meta_all = np.concatenate(y_meta_all)

# Meta-model training
meta_model = XGBRegressor(**xgb_meta_params)
meta_model.fit(X_meta_all, y_meta_all)
final_pred = meta_model.predict(X_meta_all)

In [8]:
# Final metrics on OOF predictions
print("\nMeta-Model Evaluation (XGB on OOF predictions):")
meta_mse = mean_squared_error(y_meta_all, final_pred)
meta_rmse = np.sqrt(meta_mse)
meta_mae = mean_absolute_error(y_meta_all, final_pred)
meta_r2 = r2_score(y_meta_all, final_pred)
meta_mape = np.mean(np.abs((y_meta_all - final_pred) / y_meta_all)) * 100
meta_naive = np.roll(y_meta_all, 1)
meta_naive[0] = y_meta_all[0]
meta_mase = meta_mae / np.mean(np.abs(y_meta_all - meta_naive)) if np.mean(np.abs(y_meta_all - meta_naive)) != 0 else np.nan
try:
    meta_msle = mean_squared_log_error(np.maximum(y_meta_all, 1e-6), np.maximum(final_pred, 1e-6))
except:
    meta_msle = np.nan
meta_mbe = np.mean(final_pred - y_meta_all)
meta_pcc, _ = pearsonr(y_meta_all, final_pred)
meta_srcc, _ = spearmanr(y_meta_all, final_pred)
meta_y_mean = np.mean(y_meta_all)
meta_pred_mean = np.mean(final_pred)
meta_cov = np.mean((y_meta_all - meta_y_mean) * (final_pred - meta_pred_mean))
meta_ccc = (2 * meta_cov) / (np.var(y_meta_all) + np.var(final_pred) + (meta_y_mean - meta_pred_mean) ** 2)


Meta-Model Evaluation (XGB on OOF predictions):


In [9]:
print(f"RMSE: {meta_rmse:.4f}")
print(f"MSE: {meta_mse:.4f}")
print(f"MAE: {meta_mae:.4f}")
print(f"R2: {meta_r2:.4f}")
print(f"MAPE: {meta_mape:.4f}")
print(f"MASE: {meta_mase:.4f}")
print(f"MSLE: {meta_msle:.4f}")
print(f"MBE: {meta_mbe:.4f}")
print(f"Pearson's r (PCC): {meta_pcc:.4f}")
print(f"Spearman's ρ (SRCC): {meta_srcc:.4f}")
print(f"CCC: {meta_ccc:.4f}")

RMSE: 0.0277
MSE: 0.0008
MAE: 0.0205
R2: 0.9379
MAPE: 2.6164
MASE: 0.3364
MSLE: 0.0002
MBE: 0.0000
Pearson's r (PCC): 0.9737
Spearman's ρ (SRCC): 0.9686
CCC: 0.9648
